# scPertEval — DRF + DEG figure reproduction (end-to-end)

Reproduces the DRF table, the per-metric timing table, the DEG-Jaccard composite, and the
DEG-concordance composite from the scPertEval preprint — plus the **overestim DEG variants**
(scanpy's conservative `t-test_overestim_var` vs MWU).

The datasets are downloaded **already preprocessed** from the public GCS bucket
(`gs://scperteval/processed`) — log-normalized `X`, a `perturbation` obs column, and a
`control` label; see the bucket's `README.md` for the format. Preprocessing is out of scope
for this notebook.

Pipeline: (1) install [scPertEval 0.1.0 from PyPI](https://pypi.org/project/scperteval/),
(2) download the datasets, (3) run DRF calibration for the 7 datasets + DE export for the 4 DEG
datasets (t-test, MWU, and the overestim variant), (4) render and display the figures.
Deterministic (seed 42, 8192 subsample).

> **GCP is read-only for this notebook — running it never modifies the bucket.** All GCS
> access here is download-only (`gsutil -m cp` from a public bucket); nothing is ever written
> back.

## Setup — install scPertEval

In [ ]:
import subprocess, sys
from pathlib import Path

HERE = Path.cwd()                      # run this notebook from the scPertEval-figures repo root

# scPertEval from PyPI (pulls anndata/scanpy/illico/... deps), with the `sinkhorn` extra: torch and
# geomloss are optional there, and without them `-p all` skips the two sinkhorn_w2_* protocols —
# which are rows of the DRF table. Plus matplotlib for the figure renderers.
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'scperteval[sinkhorn]==0.1.0', 'matplotlib'], check=True)
PY = sys.executable                    # everything runs through this kernel's Python

## Configuration

In [ ]:
FIG_ROOT = HERE
DATA_DIR = FIG_ROOT / 'data'           # downloaded datasets land here
DRF_OUT  = FIG_ROOT / 'drf_outputs'
DE_OUT   = FIG_ROOT / 'de_outputs'
for d in (DATA_DIR, DRF_OUT, DE_OUT): d.mkdir(exist_ok=True)

# Public, read-only bucket of preprocessed datasets. We only ever download from it.
BUCKET = 'gs://scperteval/processed'
def dataset(ds): return DATA_DIR / f'{ds}_processed_complete.h5ad'

def run(cmd, cwd=None):
    print('$', ' '.join(str(c) for c in cmd))
    subprocess.run([str(c) for c in cmd], cwd=cwd, check=True)

## Step 1 — download the datasets

Pull the standardized (already-preprocessed) `.h5ad` files from the **public** bucket into
`DATA_DIR`. Requires the gcloud SDK / `gsutil` on `PATH`. This is a download only — the
bucket is read-only and is never modified.

In [ ]:
run(['gsutil', '-m', 'cp', f'{BUCKET}/*_processed_complete.h5ad', f'{DATA_DIR}/'])

## Step 2 — scPertEval DRF for the 7 datasets

`scperteval calibrate -p all --calibrator drf --profile` computes all 22 protocols + a
per-protocol timing table per dataset, writing one per-perturbation `__drf.csv` and one
`__timing.csv` each. **Long-running** (Sinkhorn dominates).

In [ ]:
DRF_DATASETS = ['wessels23', 'replogle22rpe1', 'replogle22k562',
                'nadig25jurkat', 'nadig25hepg2', 'arch1', 'kaden25rpe1']
for ds in DRF_DATASETS:
    print(f'=== DRF {ds} ===')
    run([PY, '-m', 'scperteval', 'calibrate', dataset(ds), '-p', 'all', '--profile',
         '--subsample', 8192, '--seed', 42, '--calibrator', 'drf', '--out-dir', DRF_OUT, '--quiet'])

## Step 3 — scPertEval DE export for the 4 DEG datasets

`scperteval de` recomputes per-gene DE (GT first-half vs the 8192 pool) to HDF5, **one method
per call and per file**, so each DEG dataset needs a `t-test` pass and an `MWU` pass. A third
pass exports scanpy's conservative `t-test_overestim_var` (into `de_cache_overestim/`) for the
overestim DEG variants; those reuse the `MWU` export from `de_outputs/`, since the MWU channel
is the same DE either way.

In [ ]:
DE_DATASETS = ['wessels23', 'replogle22k562', 'nadig25hepg2', 'arch1']

# Conservative-variance DE ('t-test_overestim_var') goes to de_cache_overestim/, where the
# *_overestim_figure.py scripts look — no collision with the standard t-test in de_outputs/.
DE_OVR = FIG_ROOT / 'de_cache_overestim'; DE_OVR.mkdir(exist_ok=True)

def de(ds, method, out_dir):
    run([PY, '-m', 'scperteval', 'de', dataset(ds), '--method', method,
         '--subsample', 8192, '--seed', 42, '--out-dir', out_dir])

for ds in DE_DATASETS:
    print(f'=== DE {ds} ===')
    de(ds, 't-test', DE_OUT)
    de(ds, 'MWU', DE_OUT)
    de(ds, 't-test_overestim_var', DE_OVR)

## Step 4 — render the figures

In [ ]:
for script in ['drf_table_figure.py', 'timing_table.py',
               'deg_jaccard_figure.py', 'deg_concordance_figure.py',
               'deg_jaccard_overestim_figure.py', 'deg_concordance_overestim_figure.py']:
    run([PY, FIG_ROOT / script], cwd=FIG_ROOT)

## Step 5 — display the figures

In [ ]:
from IPython.display import Image, display
for fig in ['drf_table_mean.png', 'drf_table_median.png',
            'metric_timing_table.png', 'deg_jaccard_with_counts.png',
            'deg_concordance.png',
            'deg_jaccard_overestim_with_counts.png', 'deg_concordance_overestim.png']:
    print(fig)
    display(Image(filename=str(FIG_ROOT / 'figures' / fig)))